# __EXPERIMENT__ — <one-line title>

| Field | Value |
|---|---|
| **Version** | `__EXPERIMENT__` |
| **Plan group** | A1–E5 (see `.claude/rules/project-rules.md`, experiment plan) |
| **Parent version** | the vNNN this builds on, or — |
| **Author** | |
| **Date** | |
| **Status** | running / kept / discarded / shortlisted / submitted |

Created by `make experiment NAME=<slug>`. House rules for this notebook:

* every code cell is preceded by a markdown cell saying **what** it does and **why**;
* every function has a docstring, and non-obvious lines have a comment;
* the primary metric is **macro F0.5 on the fixed validation fold**; nothing else
  decides whether an idea is kept;
* code a second notebook needs moves to `src/entity_resolution/` (tested, documented).

## 1. Hypothesis

* **Change vs parent:** …
* **Why it should raise macro F0.5:** … (fewer false merges? more true pairs kept?)
* **Expected effect:** candidate recall …, precision …, runtime …
* **Discard if:** local F0.5 does not beat the parent by more than 0.002.

## 2. Setup

Imports from the shared library, the fixed seed, and this experiment's folder.
`EXP_DIR` receives `metrics.json`; `ARTIFACTS` (gitignored) holds models, candidate
sets and predictions. `timings` collects stage run times for the log.

In [ ]:
import numpy as np
import pandas as pd

from entity_resolution import config as C
from entity_resolution.metrics import breakdown, candidate_report
from entity_resolution.split import load_fold
from entity_resolution.tracking import log_result, timed

EXP_DIR = C.EXPERIMENTS / "__EXPERIMENT__"
ARTIFACTS = EXP_DIR / "artifacts"
rng = np.random.default_rng(C.SEED)  # every random choice draws from this
timings: dict[str, float] = {}

## 3. Data

The fixed validation split (`entity_resolution.split`): every experiment scores the
same held-out Source 1 entities against the same Source 2/3 pool, so local F0.5 values
are comparable across versions. Fit anything trainable on the `train` fold only.
Pass `columns=[...]` to load less; a full fold loads in a few seconds from the cache
(`make cache` once per machine).

In [ ]:
with timed("load", timings):
    val = load_fold("val")
    train = load_fold("train")  # drop if this experiment trains nothing
pd.DataFrame([val.summary(), train.summary()])

## 4. Method

One subsection per pipeline stage. Say what each stage does, why this choice, and
what it costs. Delete the stages this experiment does not touch.

### 4.1 Normalisation

Rules applied to names and addresses (case, accents, punctuation, legal suffixes,
abbreviations, transliteration) and the evidence from the data for each.

In [ ]:
def normalise(df: pd.DataFrame) -> pd.DataFrame:
    """Return ``df`` with normalised name and address columns added.

    Document every rule here and why it helps matching.
    """
    raise NotImplementedError

### 4.2 Blocking (candidate generation)

Produces `candidates`: Source 1 id → list of Source 2/3 ids worth scoring. Blocking
caps recall, so section 5 reports `candidate_report` before any matcher is judged.

In [ ]:
def block(s1: pd.DataFrame, pool: pd.DataFrame) -> dict[str, list[str]]:
    """Candidate Source 2/3 ids for every Source 1 entity in ``s1``."""
    raise NotImplementedError

### 4.3 Pair features

One row per (Source 1, candidate) pair. List each feature and what noise it targets.

In [ ]:
def pair_features(pairs: pd.DataFrame) -> pd.DataFrame:
    """Similarity features for candidate pairs (one row per pair)."""
    raise NotImplementedError

### 4.4 Matching model

Model, why this one, hyper-parameters, training data (train fold only), licence
(MIT/Apache-2.0, at most 8B parameters for any pretrained model).

In [ ]:
def fit_matcher(features: pd.DataFrame, labels: pd.Series):
    """Fit the pairwise matcher on train-fold pairs and return it."""
    raise NotImplementedError

### 4.5 Decision rule

How pair scores become each entity's final list: threshold, score gap, singleton
handling, multi-match selection. F0.5 punishes false merges twice as hard as misses.

In [ ]:
def decide(scored_pairs: pd.DataFrame) -> dict[str, list[str]]:
    """Final matched ids per Source 1 entity from scored candidate pairs."""
    raise NotImplementedError

## 5. Evaluation on the validation fold

Primary metric: **macro F0.5** over validation Source 1 entities, singletons included
(`breakdown`). Blocking quality: `candidate_report` (pair recall, F0.5 ceiling,
candidates per entity). Accuracy, AUC and F1 are diagnostics only.

In [ ]:
truth = val.truth()
pool = pd.concat([val.s2, val.s3], ignore_index=True)
with timed("blocking", timings):
    candidates = block(val.s1, pool)
blocking = candidate_report(candidates, truth, pool_size=len(pool))
with timed("matching", timings):
    matches = decide(...)  # score the candidate pairs with the fitted matcher
scores = breakdown(matches, truth)
pd.Series({**blocking, **scores, **timings})

## 6. Error analysis

Inspect the worst entities: false merges (they hurt most under F0.5), false merges
on singletons, and missed matches. Show 10–20 examples of each and name the pattern;
the patterns drive the next experiment.

## 7. Log the result

Records `metrics.json` and this version's row in `experiments/experiments.csv`,
stamped with the git commit of `src/`. Commit the notebook (with outputs) right after.

In [ ]:
log_result(
    EXP_DIR,
    change="…",  # one line: what differs from the parent version
    group="A1",  # plan ID, A1–E5
    local_f05=scores["f_beta"],
    cand_recall=blocking["pair_recall"],
    notes="…",
    metrics={**blocking, **scores, **timings},
)

## 8. Conclusion

* **Result vs parent:** local F0.5 … → … (Δ …)
* **Keep or discard:** …
* **Next experiment:** …

## 9. Test inference (shortlisted versions only)

Only for a version picked for a leaderboard upload. Load the test sources, run the
same pipeline, then write and validate both files:

```python
from entity_resolution.data import load_sources
from entity_resolution.submission import write_submission
test = load_sources("test")
write_submission(matches_test, candidates_test, test[1][C.ENTITY_ID].tolist())
```

Then `make validate`, upload `output/matching_results.tsv`, add the entry to
`LEADERBOARD.md`, and record the score: `make public V=vNNN SCORE=0.xxxx`.